In [ ]:
import anndata

import numpy as np
import pandas as pd
import scanpy as sc

import infercnvpy as cnv


sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, facecolor='white', frameon=False,  figsize=[4,4])

In [ ]:
adata_epithelial = sc.read_h5ad(filename='../yourpath/2_2_atlas_anno_coarse.h5ad')
adata_epithelial = adata_epithelial[adata_epithelial.obs['anno_atlas_coarse'] == "Epithelial cells"]
adata_epithelial.layers["counts"] = adata_epithelial.X.copy()
sc.pp.normalize_total(adata_epithelial, target_sum=1e4)
sc.pp.log1p(adata_epithelial)
adata_epithelial.raw = adata_epithelial
adata_epithelial

In [ ]:
sc.pp.neighbors(adata_epithelial, use_rep='X_scANVI')
sc.tl.umap(adata_epithelial)
sc.tl.leiden(adata_epithelial, resolution=1)

In [ ]:
gtf = "../../../../yourpath/gencode.v49.annotation.gtf"
cnv.io.genomic_position_from_gtf(gtf_file=gtf, adata=adata_epithelial, gtf_gene_id='gene_name')

In [ ]:
ref_key = "leiden"
ref_cats = ["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25","26"] 

cnv.tl.infercnv(
    adata_epithelial,
    reference_key=ref_key,
    reference_cat=ref_cats,
    window_size=100,
    step=10,
    exclude_chromosomes=("chrX","chrY"),
    key_added="cnv"
)

In [14]:
cnv.pp.neighbors(adata_epithelial, use_rep="cnv")
cnv.tl.leiden(adata_epithelial, resolution=1.0, key_added="cnv_leiden")
cnv.tl.cnv_score(adata_epithelial, groupby="cnv_leiden", use_rep="cnv", key_added="cnv_score")

In [ ]:
adata_epithelial.write('../yourpath/3_infercnv_adata_epithelial.h5ad')